# Claude로 SQL query를 만드는 방법


이 notebook에서는 자연어 질문을 바탕으로 SQL query를 생성하는 데 Claude를 사용하는 방법을 살펴봅니다. 테스트 database를 설정하고 Claude에 schema를 제공하며, 사람의 언어를 이해하고 SQL query로 변환하는 모습을 보여 줍니다.

## 설정

먼저 필요한 library를 설치하고 API key로 Anthropic client를 설정합니다.

In [ ]:
# 필요한 library 설치
%pip install anthropic

In [2]:
# 필요한 library import
import sqlite3

from anthropic import Anthropic

# Claude API client 설정
client = Anthropic()
MODEL_NAME = "claude-opus-4-8"

## 테스트 Database 생성

SQLite로 테스트 database를 만들고 sample data를 채웁니다.

In [3]:
# 테스트 database에 연결(없으면 생성)
conn = sqlite3.connect("test_db.db")
cursor = conn.cursor()

# sample table 생성
cursor.execute("""
    CREATE TABLE IF NOT EXISTS employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        department TEXT,
        salary INTEGER
    )
""")

# sample data 삽입
sample_data = [
    (1, "John Doe", "Sales", 50000),
    (2, "Jane Smith", "Engineering", 75000),
    (3, "Mike Johnson", "Sales", 60000),
    (4, "Emily Brown", "Engineering", 80000),
    (5, "David Lee", "Marketing", 55000),
]
cursor.executemany("INSERT INTO employees VALUES (?, ?, ?, ?)", sample_data)
conn.commit()

## Claude로 SQL Query 생성

이제 자연어 질문을 Claude에 보내고 생성된 SQL query를 받는 함수를 정의합니다.

In [8]:
# Claude에 query를 보내고 응답을 받는 함수 정의
def ask_claude(query, schema):
    prompt = f"""Here is the schema for a database:

{schema}

Given this schema, can you output a SQL query to answer the following question? Only output the SQL query and nothing else.

Question: {query}
"""

    response = client.messages.create(
        model=MODEL_NAME, max_tokens=2048, messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

database schema를 조회하고 문자열로 형식을 지정합니다.

In [5]:
# database schema 얻기
schema = cursor.execute("PRAGMA table_info(employees)").fetchall()
schema_str = (
    "CREATE TABLE EMPLOYEES (\n" + "\n".join([f"{col[1]} {col[2]}" for col in schema]) + "\n)"
)
print(schema_str)

CREATE TABLE EMPLOYEES (
id INTEGER
name TEXT
department TEXT
salary INTEGER
)


이제 자연어 질문 예제를 제공하고 Claude에 보내겠습니다.

In [9]:
# 자연어 질문 예제
question = "What are the names and salaries of employees in the Engineering department?"
# Claude에 질문을 보내고 SQL query 받기
sql_query = ask_claude(question, schema_str)
print(sql_query)

SELECT name, salary
FROM EMPLOYEES
WHERE department = 'Engineering';


## 생성된 SQL Query 실행

마지막으로 생성된 SQL query를 테스트 database에서 실행하고 결과를 출력합니다.

In [10]:
# SQL query를 실행하고 결과 출력
results = cursor.execute(sql_query).fetchall()

for row in results:
    print(row)

('Jane Smith', 75000)
('Emily Brown', 80000)


작업을 마치면 database connection을 닫는 것을 잊지 마세요.

In [11]:
# database connection 닫기
conn.close()